<a href="https://colab.research.google.com/github/vifirsanova/ML-2026-pt-2/blob/main/tutorials/3_experiments/experimental_canvas.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Как провести простое и быстрое исследование

Здесь мы рассмотрим минимальный пример бенчмаркинга инструментов для инференса LLM

**Постановка задачи**

- Сравнительный анализ производительности систем инференса vLLM и HF Transformers без оптимизации PagedAttention и FlashAttention

> При выборе формулировки мы определяем алгоритм, требования к hardware, формальную задачу и юз-кейсы, а также параметр, который мы будем сравнивать в одном и том же сетапе + бенчмарк и метрики оценки

Чек-лист для проведения исследования:

- Бейзлайн (какой алгоритм мы исследуем с целью улучшения его показателей на решаемой задаче)
- Формальное описание задачи (в идеале в формулах, например, в терминах теории вероятностей)
- Сферы применения (формальная задача - это то, что решает нейронка, а в каких инструментах применяется такая нейронка?)
- Метрики оценки (как стандартные метрики качества, так и замеры скорости вычислений, требования к памяти)
- Параметры сравнения (мы улучшаем бейзлайн, как мы модифицируем исходный алгоритм?)

Уровни изменения архитектуры:
- минимум: меняем гиперпараметры, выбираем оптимальный сетап (например, поменяли количетсов эпох, размер батча)
- рекомендуемый уровень: добавление/изменение слоев, количества нейронов, метода оптимизации (заменили DPO на KTO, потому что у нас нет возможности собрать пары предпочтений для дообучения модели, а KTO позволяет произвести оптимизацию без них)
- профессиональный уровень: изменение логики работы алгоритма а уровне вычислений (вспомним статью про DPO: исследователи проанализировали math behind PPO, изменили формулу так, чтобы можно посчитать результат без Reward Model)

Изменения всегда точечные и хорошо обоснованные с точки зрения оптимизации решения (у нас нет пар данных, поэтому KTO; у нас мало ресурсов, поэтому QLoRA, а не LoRA), подстановки формул (сокращение, возможно решение с помощью другой статистической или вероятностной модели), совместимости с текущим решением (возможные изменения гиперпараметров)

**Контрибьюшн**

- Определение оптимальных сценариев использования каждого инструмента на основе эмпирических данных

- Разработка системы инференса для разных исследовательских задач

> Как определить, что я накотрибьютил?

Чек-лист:

- Датасет: использован существующий или создавался новый?
- Метрика: использована стандартная или предложена новая (или модификация существующей, например, взвешенный вариант F-Score для вашей задачи, новый метод пенализации повторов генеративной модели, human-in-the-loop)
- Алгоритм: все, что вы тюнили, представляет собой новую модель
- Инфраструктура для тестирования, библиотека, любая reproducibility

**Результаты исследования**

Мы доказали, что выбор движка для оптимизации обусловлен характеристиками фреймворка, а не максимальными возможностями данного алгоритма.

Что это даёт для разработчика: takeaway -> если стоит выбор между vLLM и бейзлановым HF Transformers, сначала нужно определить инфраструктуру разрабатываемого решения. Если предлагается работа с диалогами, многопоточоность -> vLLM. Если это однократная операция, отладка, ресерч -> HF Transformers (vLLM даст худшие показатели из-за сложности архитектуры)

Как мы это поняли?

**Экспериментальный сетап**

1. Одинаковые условия:
  - однократный вызов модели
  - имитация многопользовательского сценария

> Мы создаем эмулятор для двух сценариев путем синтеза данных или выбора релевантных датасетов

2. Архитектурный сетап:
  - одна и та же модель
  - один и тот же датасет
  - различие: запуск через vLLM, запуск через HF Transformers

3. Тестирование:
  - разработка инфраструктуры на основе свойств vLLM, которые заявлены авторами (что улучшает PagedAttention+FlashAttention: скорость и управление памятью)
  - нас интересует скорость вычислений и затраты памяти во время инференса
  - метрики оценки качества должны соответствовать датасету, модели, формальной задаче
  - экспериментальная инфраструктура должна выдавать отчет по метрикам для каждого варианта сетапа


**Исследовательский вопрос и гипотеза**

Есть ли разница в производительности моделей, построенных с помощью разных инференс-провайдеров в однопользовательских и многопользовательских сценариях?

H0: vLLM лучше в любых сценариях
H1: Transformers лучше в любых сценариях  
H2: Зависит от сценария (vLLM лучше в многопользовательских архитектурно, но может проигрывать в однопользовательских сценариях из-за "перегруженности")

#### Пример использования

In [ ]:
#!pip install vllm

from vllm import LLM, SamplingParams

# Sample data
prompts = ["Explain AI in"] * 10

# vLLM
llm = LLM(model="facebook/opt-125m")
outputs = llm.generate(prompts, SamplingParams(temperature=0.8, max_tokens=50))

# Transformers pipeline
from transformers import pipeline
pipe = pipeline("text-generation", model="facebook/opt-125m")
result = pipe(prompts, max_length=50)

INFO 11-15 04:45:29 [__init__.py:216] Automatically detected platform cuda.
INFO 11-15 04:45:49 [utils.py:233] non-default args: {'disable_log_stats': True, 'model': 'facebook/opt-125m'}


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 11-15 04:46:08 [model.py:547] Resolved architecture: OPTForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


...

Adding requests:   0%|          | 0/10 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/10 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

Device set to use cuda:0


model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

Truncation was not explicitly activated but `max_length` is provided a specific value, please use `truncation=True` to explicitly truncate examples to max length. Defaulting to 'longest_first' truncation strategy. If you encode pairs of sequences (GLUE-style) with the tokenizer you can select this strategy more precisely by providing a specific strategy to `truncation`.
Both `max_new_tokens` (=256) and `max_length`(=51) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=51) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=256) and `max_length`(=51) seem to have been set. `max_new_tokens` will take precedence. Please refer to 

In [ ]:
result # Transformers pipeline

[[{'generated_text': 'Explain AI in the title. This is a dumb question.\nThis is a stupid question.'}],
 [{'generated_text': "Explain AI in my head.\nI'm not sure why they would ask that in a video game, it's really not that different from most other video games, but they would ask for a specific AI.  I've seen a lot of people saying that games are going to have a lot of AI in them but I'm not seeing any games that are going to use AI. I don't think there is a particular AI in a game, but it's something that's really hard to explain with a video game.\nI dont think there is an AI in a game, the AI in the game is in game, but not in the game.\nTrue but I think it's harder to explain. I guess the AI in games is in game, but the AI in real life is not in game, they're just talking about the game."}],
 [{'generated_text': "Explain AI in the title please!\nI think it's the title of the video.\nAfaik the title is a picture of a robot, but I don't know if it's intentional or not."}],
 [{'gene

In [ ]:
outputs #vLLM

[RequestOutput(request_id=0, prompt='Explain AI in', prompt_token_ids=[2, 43043, 1851, 4687, 11], encoder_prompt=None, encoder_prompt_token_ids=None, prompt_logprobs=None, outputs=[CompletionOutput(index=0, text=" terms of facts and statistics. I am still learning that.\nYou can't just point out statistics and statistics without explaining how decisions are made. Also you can't just say AI is the same as the fact that people are smarter.", token_ids=[1110, 9, 4905, 8, 6732, 4, 38, 524, 202, 2239, 14, 4, 50118, 1185, 64, 75, 95, 477, 66, 6732, 8, 6732, 396, 8926, 141, 2390, 32, 156, 4, 1578, 47, 64, 75, 95, 224, 4687, 16, 5, 276, 25, 5, 754, 14, 82, 32, 18369, 4, 2], cumulative_logprob=None, logprobs=None, finish_reason=stop, stop_reason=None)], finished=True, metrics=None, lora_request=None, num_cached_tokens=0, multi_modal_placeholders={}),
 RequestOutput(request_id=1, prompt='Explain AI in', prompt_token_ids=[2, 43043, 1851, 4687, 11], encoder_prompt=None, encoder_prompt_token_ids=No

#### Пример экспериментальной инфраструктуры
    
Принципы построения:
1. Модульность
2. Воспроизводимость - детерминированные условия тестирования  
3. Масштабируемость - легко добавлять новые системы и метрики
4. Хорошая документация

In [ ]:
#!pip install vllm

import time
import torch
from vllm import LLM, SamplingParams
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import pandas as pd
from typing import List, Dict
import warnings
warnings.filterwarnings('ignore')

class InferenceBenchmark:
    def __init__(self, model_name="facebook/opt-125m"):
        """
        На что обращаем внимание при инициализации:
        - можно тестировать семейства моделей разных размеров
        - разные типы промптов (диалоги, код, reasoning)
        - контроль аппаратного обеспечения: GPU/CPU
        """
        self.model_name = model_name
        # Экспериментальный датасет можно категоризировать
        # - Добавить категории промптов
        # - Разнообразить и сравнить домены, языки
        # - Смоделировать задачи (CLS, QA, NLU)
        self.prompts = [
            "Explain the concept of machine learning in simple terms:",
            "What are the benefits of renewable energy?",
            "How does the internet work?",
            "Describe the process of photosynthesis:",
            "What is artificial intelligence?",
            "Explain quantum computing basics:",
            "How do neural networks learn?",
            "What is blockchain technology?",
            "Describe the water cycle:",
            "What are the laws of thermodynamics?"
        ] * 3  # 30 prompts total for better statistics
        self.results_df = pd.DataFrame()

    def benchmark_vllm(self, num_runs: int = 3) -> pd.DataFrame:
        """Benchmark vLLM inference speed"""

        # Инициализация vLLM: фиксированные настройки или параметризация
        llm = LLM(
            model=self.model_name,
            trust_remote_code=True,
            max_model_len=512,          # лимит контекста
            gpu_memory_utilization=0.8  # использует 80% общей памяти GPU
            # For vLLM v0.7.0 and later, the vLLM instance will only use
            # gpu_memory_utilization of the total memory
            # https://verl.readthedocs.io/en/latest/perf/perf_tuning.html
        )

        sampling_params = SamplingParams(
            temperature=0.7,
            top_p=0.9,
            max_tokens=100, # влияние длины токенов на скорость вычисления (связь с KV-cache)
            stop=["\n", "###"]
        )

        run_results = []

        # Для академических экспериментов требуется
        # многократный (не менее 5 раз) запуск одного и того же сетапа (стат. значимость)
        for run in range(num_runs):
            start_time = time.time()

            # Continuous batching
            # На один запрос выделяется один батч (традиционный подход)
            # ████████████████████               <- запросы завершаются когда готовы
            # ██████████████████████████████████
            # ██████████████
            # Новый запрос -> █████████ <- добавляется в работающий батч (динамически)
            outputs = llm.generate(self.prompts, sampling_params)
            # Результат: GPU не простаивает, запрос не ждут полного батча (быстрее),
            # эффективный менеджмент ресурсов снижает latency (production-оптимизация)
            end_time = time.time()
            batch_time = end_time - start_time

            # Расчет количества токенов за секунду
            total_tokens = sum(len(output.outputs[0].token_ids) for output in outputs)
            tps = total_tokens / batch_time

            run_results.append({
                'engine': 'vLLM',
                'run': run + 1, # номер итерации (top-tier конференции требуют отчетности)
                'time_seconds': batch_time,
                'tokens_per_second': tps,
                'total_tokens': total_tokens,
                'num_requests': len(self.prompts)
                # также важно фиксировать параметры LLM/vLLM:
                # gpu_mem: 0.8, batching: continuos, etc.
            })

        return pd.DataFrame(run_results)

    def benchmark_transformers(self, num_runs: int = 3) -> pd.DataFrame:
        """
        Тестирование стандартного Hugging Face Transformers
        Очевидное ограничение: статический батчинг
        ████████████████████████████████████
                                         ████████████████████████████████████
        Батчинг - проблема библиотеки и проблема нашего исследования.
        При формировании проблемы важно найти бутылочное горлышко
        существующих решений.

        Этот подход представляет baseline - наивную реализацию без оптимизаций.
        Используется для демонстрации выигрыша от разработанных систем.

        Сравнить систему с бейзлайном - только первый этап, первичная проверка гипотезы.
        Далее мы сравниваем результат с вариантами реализации нашей модели и с конкурирующими решениями.

        Архитектурные ограничения, которые мы хотим решить:
        - Каждый запрос обрабатывается отдельно
        - Полные матрицы внимания без оптимизации памяти
        - Фиксированные размеры батчей

        Для исследовательских целей можно:
        - Исследовать типы оптимизации (gradient checkpointing, mixed precision)
        - Протестировать разные реализации внимания (flash, memory-efficient)
        """

        # Инициализация стандартного пайплайна - подход "из коробки"
        pipe = pipeline(
            "text-generation",
            model=self.model_name,
            tokenizer=self.model_name,
            device="cuda" if torch.cuda.is_available() else "cpu",
            torch_dtype=torch.float16, # параметры (или наборы параметров) должны быть совместимы с другими сетапами
            max_new_tokens=100,
            temperature=0.7,
            do_sample=True,
            pad_token_id=50256
        )

        run_results = []

        for run in range(num_runs):
            start_time = time.time()

            for prompt in self.prompts:
                result = pipe(
                    prompt,
                    max_new_tokens=100,
                    temperature=0.7,
                    do_sample=True
                )

            end_time = time.time()
            batch_time = end_time - start_time

            # Здесь нужно реализовать расчет на уровне токенов
            total_tokens = len(self.prompts) * 100
            tps = total_tokens / batch_time

            run_results.append({
                'engine': 'Transformers',
                'run': run + 1,
                'time_seconds': batch_time,
                'tokens_per_second': tps,
                'total_tokens': total_tokens,
                'num_requests': len(self.prompts)
            })

        return pd.DataFrame(run_results)

    def benchmark_transformers_batch(self, num_runs: int = 3) -> pd.DataFrame:
        """
        На основе изучения теории мы выяснили, что основным ограничением Transformers
        будет батчинг, поэтому здесь мы рассмотрим вариант реализации другого типа
        батчинга вручную.

        Static batching pattern:
        ████████████████████████████████████
        ████████████████████████████████████ <- все запросы одинаковой длины
        ████████████████████████████████████ <- завершаются одновременно

        Это учебный пример, но на практике эвристики и интуитивно понятные примеры
        могут разрешить даже серьезные ограничения библиотек.

        НО: Все запросы должны иметь одинаковую длину (или быть дополнены padding)
        И: Все запросы завершаются одновременно, даже если некоторые могли закончиться раньше

        В отличие от vLLM, памятью управляет PyTorch, что дает возможности для масштабирования:
        - Исследовать optimal batch size для разных моделей, динамические подходы
        - Тестировать различные стратегии padding (left/right/longest), bucketing
        - Экспериментировать с разными dtype (float16/bfloat16/int8)
        """

        tokenizer = AutoTokenizer.from_pretrained(self.model_name)
        model = AutoModelForCausalLM.from_pretrained(
            self.model_name,
            torch_dtype=torch.float16,
            device_map="auto"
        )

        if tokenizer.pad_token is None:
            tokenizer.pad_token = tokenizer.eos_token

        run_results = []

        for run in range(num_runs):
            start_time = time.time()

            # Батчевая токенизация
            # Ограничение: требует padding -> избыточные вычисления

            # Пример padding overhead:
            # Запрос 1: "Hello" (5 токенов) -> padding до 10 токенов создает 5 лишних токенов
            # Запрос 2: "Hello world" (10 токенов)
            inputs = tokenizer(
                self.prompts,
                padding=True,
                truncation=True,
                max_length=512,
                return_tensors="pt"
            ).to(model.device)

            with torch.no_grad():
                outputs = model.generate(
                    **inputs,
                    max_new_tokens=100,
                    temperature=0.7,
                    do_sample=True,
                    pad_token_id=tokenizer.pad_token_id,
                    eos_token_id=tokenizer.eos_token_id
                )

            end_time = time.time()
            batch_time = end_time - start_time

            total_tokens = outputs.numel() # подсчет токенов с учетом паддинга (элементов в тензоре)
            tps = total_tokens / batch_time

            run_results.append({
                'engine': 'Transformers-Batch',
                'run': run + 1,
                'time_seconds': batch_time,
                'tokens_per_second': tps,
                'total_tokens': total_tokens,
                'num_requests': len(self.prompts)
            })

        return pd.DataFrame(run_results)

    def run_comprehensive_benchmark(self) -> pd.DataFrame:
        """
        1. Запуск всех тестов в изолированных условиях
        2. Агрегация результатов
        3. Статистический анализ
        4. Генерация отчетов

        Для промышленного использования можно добавить трекинг MLflow, W&B
        """
        vllm_results = self.benchmark_vllm()
        transformers_batch_results = self.benchmark_transformers_batch()
        transformers_results = self.benchmark_transformers()

        all_results = pd.concat([
            vllm_results,
            transformers_batch_results,
            transformers_results
        ], ignore_index=True)

        summary = all_results.groupby('engine').agg({
            'time_seconds': ['mean', 'min', 'max', 'std'],
            'tokens_per_second': ['mean', 'min', 'max', 'std'],
            'total_tokens': 'mean',
            'num_requests': 'mean'
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()

        transformers_tps = summary[summary['engine'] == 'Transformers']['tokens_per_second_mean'].values[0]
        transformers_batch_tps = summary[summary['engine'] == 'Transformers-Batch']['tokens_per_second_mean'].values[0]

        speedup_data = []
        for _, row in summary.iterrows():
            speedup_vs_transformers = row['tokens_per_second_mean'] / transformers_tps
            speedup_vs_batch = row['tokens_per_second_mean'] / transformers_batch_tps

            speedup_data.append({
                'engine': row['engine'],
                'time_seconds_mean': row['time_seconds_mean'],
                'tokens_per_second_mean': row['tokens_per_second_mean'],
                'time_seconds_std': row['time_seconds_std'],
                'tokens_per_second_std': row['tokens_per_second_std'],
                'speedup_vs_transformers': round(speedup_vs_transformers, 2),
                'speedup_vs_batch': round(speedup_vs_batch, 2) if row['engine'] != 'Transformers-Batch' else None
            })

        final_summary = pd.DataFrame(speedup_data)
        self.results_df = final_summary

        return final_summary

if __name__ == "__main__":
    benchmark = InferenceBenchmark("facebook/opt-125m")
    results_df = benchmark.run_comprehensive_benchmark()

    print("Inference Benchmark Results:")
    print("=" * 80)
    print(results_df.to_string(index=False))

    results_df.to_csv("inference_benchmark_results.csv", index=False)

INFO 11-15 05:01:45 [__init__.py:216] Automatically detected platform cuda.
INFO 11-15 05:02:10 [utils.py:233] non-default args: {'trust_remote_code': True, 'max_model_len': 512, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'facebook/opt-125m'}


The argument `trust_remote_code` is to be used with Auto classes. It has no effect here and is ignored.


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 11-15 05:02:42 [model.py:547] Resolved architecture: OPTForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


...

Processed prompts:   0%|          | 0/30 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.


model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
A decoder-only architecture is being used, but right-padding was detected! For correct generation results, please set `padding_side='left'` when initializing the tokenizer.
`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda
You seem to be using the pipelines sequentially on GPU. In order to maximize efficiency please use a dataset


Inference Benchmark Results:
            engine  time_seconds_mean  tokens_per_second_mean  time_seconds_std  tokens_per_second_std  speedup_vs_transformers  speedup_vs_batch
      Transformers              25.81                  116.37              1.10                   4.98                     1.00              0.05
Transformers-Batch               2.55                 2314.76              2.58                1477.69                    19.89               NaN
              vLLM              10.71                  122.09             12.93                 140.10                     1.05              0.05


In [ ]:
#!pip install vllm

import time
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForCausalLM
import pandas as pd
import warnings
warnings.filterwarnings('ignore')

class InferenceBenchmark:
    def __init__(self, model_name="facebook/opt-125m"):
        self.model_name = model_name
        self.prompts = [
            "Explain machine learning:",
            "What is AI?",
            "How does internet work?",
            "Describe photosynthesis:",
            "What is blockchain?",
        ] * 30  # 150 prompts
        self.results_df = pd.DataFrame()
        self.vllm_available = self._check_vllm_availability()

    def _check_vllm_availability(self) -> bool:
        try:
            from vllm import LLM, SamplingParams
            return True
        except Exception as e:
            print(f"vLLM not available: {e}")
            return False

    def benchmark_vllm(self, num_runs: int = 2) -> pd.DataFrame:
        if not self.vllm_available:
            return pd.DataFrame()

        try:
            from vllm import LLM, SamplingParams

            llm = LLM(
                model=self.model_name,
                max_model_len=512,
                gpu_memory_utilization=0.8,
            )

            sampling_params = SamplingParams(
                temperature=0.7,
                top_p=0.9,
                max_tokens=100,
            )

            run_results = []

            for run in range(num_runs):
                start_time = time.time()

                outputs = llm.generate(self.prompts, sampling_params)
                end_time = time.time()
                batch_time = end_time - start_time

                total_tokens = sum(len(output.outputs[0].token_ids) for output in outputs)
                tps = total_tokens / batch_time

                run_results.append({
                    'engine': 'vLLM',
                    'run': run + 1,
                    'time_seconds': round(batch_time, 2),
                    'tokens_per_second': round(tps, 1),
                    'total_tokens': total_tokens,
                    'num_requests': len(self.prompts)
                })

            return pd.DataFrame(run_results)

        except Exception as e:
            print(f"vLLM failed: {e}")
            return pd.DataFrame()

    def benchmark_transformers(self, num_runs: int = 2) -> pd.DataFrame:
        run_results = []

        try:
            # Убираем max_length чтобы избежать конфликта
            pipe = pipeline(
                "text-generation",
                model=self.model_name,
                device="cuda" if torch.cuda.is_available() else "cpu",
                torch_dtype=torch.float16,
                max_new_tokens=100,  # ТОЛЬКО max_new_tokens
                temperature=0.7,
                do_sample=True,
            )

            for run in range(num_runs):
                start_time = time.time()

                for prompt in self.prompts:
                    result = pipe(prompt)

                end_time = time.time()
                batch_time = end_time - start_time

                total_tokens = len(self.prompts) * 100
                tps = total_tokens / batch_time

                run_results.append({
                    'engine': 'Transformers',
                    'run': run + 1,
                    'time_seconds': round(batch_time, 2),
                    'tokens_per_second': round(tps, 1),
                    'total_tokens': total_tokens,
                    'num_requests': len(self.prompts)
                })

        except Exception as e:
            print(f"Transformers failed: {e}")

        return pd.DataFrame(run_results)

    def benchmark_transformers_batch(self, num_runs: int = 2) -> pd.DataFrame:
        run_results = []

        try:
            tokenizer = AutoTokenizer.from_pretrained(self.model_name, padding_side='left')
            model = AutoModelForCausalLM.from_pretrained(
                self.model_name,
                torch_dtype=torch.float16,
                device_map="auto"
            )

            if tokenizer.pad_token is None:
                tokenizer.pad_token = tokenizer.eos_token

            for run in range(num_runs):
                start_time = time.time()

                inputs = tokenizer(
                    self.prompts,
                    padding=True,
                    truncation=True,
                    max_length=512,
                    return_tensors="pt"
                ).to(model.device)

                with torch.no_grad():
                    outputs = model.generate(
                        **inputs,
                        max_new_tokens=100,  # ТОЛЬКО max_new_tokens
                        temperature=0.7,
                        do_sample=True,
                        pad_token_id=tokenizer.pad_token_id,
                    )

                end_time = time.time()
                batch_time = end_time - start_time

                total_tokens = outputs.numel()
                tps = total_tokens / batch_time

                run_results.append({
                    'engine': 'Transformers-Batch',
                    'run': run + 1,
                    'time_seconds': round(batch_time, 2),
                    'tokens_per_second': round(tps, 1),
                    'total_tokens': total_tokens,
                    'num_requests': len(self.prompts)
                })

        except Exception as e:
            print(f"Transformers-Batch failed: {e}")

        return pd.DataFrame(run_results)

    def run_comprehensive_benchmark(self) -> pd.DataFrame:
        print(f"Benchmarking with {len(self.prompts)} prompts")

        vllm_results = self.benchmark_vllm()
        transformers_batch_results = self.benchmark_transformers_batch()
        transformers_results = self.benchmark_transformers()

        all_results = pd.concat([
            vllm_results,
            transformers_batch_results,
            transformers_results
        ], ignore_index=True)

        if all_results.empty:
            print("No results collected - all benchmarks failed")
            return pd.DataFrame()

        print("Raw results:")
        print(all_results)

        summary = all_results.groupby('engine').agg({
            'time_seconds': ['mean', 'std'],
            'tokens_per_second': ['mean', 'std'],
        }).round(2)

        summary.columns = ['_'.join(col).strip() for col in summary.columns.values]
        summary = summary.reset_index()

        speedup_data = []

        for _, row in summary.iterrows():
            engine_data = {
                'engine': row['engine'],
                'time_seconds_mean': row['time_seconds_mean'],
                'tokens_per_second_mean': row['tokens_per_second_mean'],
                'time_seconds_std': row['time_seconds_std'],
                'tokens_per_second_std': row['tokens_per_second_std'],
            }
            speedup_data.append(engine_data)

        final_summary = pd.DataFrame(speedup_data)
        return final_summary

if __name__ == "__main__":
    benchmark = InferenceBenchmark("facebook/opt-125m")
    results_df = benchmark.run_comprehensive_benchmark()

    if not results_df.empty:
        print("\n" + "=" * 80)
        print("Inference Benchmark Results:")
        print("=" * 80)
        print(results_df.to_string(index=False))
    else:
        print("No results to display")

INFO 11-15 05:39:09 [__init__.py:216] Automatically detected platform cuda.
Benchmarking with 150 prompts
INFO 11-15 05:39:16 [utils.py:233] non-default args: {'max_model_len': 512, 'gpu_memory_utilization': 0.8, 'disable_log_stats': True, 'model': 'facebook/opt-125m'}


config.json:   0%|          | 0.00/651 [00:00<?, ?B/s]

INFO 11-15 05:39:45 [model.py:547] Resolved architecture: OPTForCausalLM


`torch_dtype` is deprecated! Use `dtype` instead!


INFO 11-15 05:39:45 [model.py:1510] Using max model len 512
INFO 11-15 05:39:51 [scheduler.py:205] Chunked prefill is enabled with max_num_batched_tokens=8192.


...

Adding requests:   0%|          | 0/150 [00:00<?, ?it/s]

Processed prompts:   0%|          | 0/150 [00:00<?, ?it/s, est. speed input: 0.00 toks/s, output: 0.00 toks/s]

model.safetensors:   0%|          | 0.00/251M [00:00<?, ?B/s]

`torch_dtype` is deprecated! Use `dtype` instead!
Device set to use cuda
Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)
Both `max_new_tokens` (=100) and `max_length`(=21) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://hu

Raw results:
               engine  run  time_seconds  tokens_per_second  total_tokens  \
0                vLLM    1         38.82              329.3         12783   
1                vLLM    2          8.29             1561.5         12943   
2  Transformers-Batch    1          2.32             6866.7         15900   
3  Transformers-Batch    2          2.19             7246.0         15900   
4        Transformers    1        121.72              123.2         15000   
5        Transformers    2        126.33              118.7         15000   

   num_requests  
0           150  
1           150  
2           150  
3           150  
4           150  
5           150  

Inference Benchmark Results:
            engine  time_seconds_mean  tokens_per_second_mean  time_seconds_std  tokens_per_second_std
      Transformers             124.02                  120.95              3.26                   3.18
Transformers-Batch               2.26                 7056.35              0.09      

```
               engine  run  time_seconds  tokens_per_second  total_tokens  \
0                vLLM    1         38.82              329.3         12783   
1                vLLM    2          8.29             1561.5         12943   
2  Transformers-Batch    1          2.32             6866.7         15900   
3  Transformers-Batch    2          2.19             7246.0         15900   
4        Transformers    1        121.72              123.2         15000   
5        Transformers    2        126.33              118.7         15000   

   num_requests  
0           150  
1           150  
2           150  
3           150  
4           150  
5           150  

================================================================================
Inference Benchmark Results:
================================================================================
            engine  time_seconds_mean  tokens_per_second_mean  time_seconds_std  tokens_per_second_std
      Transformers             124.02                  120.95              3.26                   3.18
Transformers-Batch               2.26                 7056.35              0.09                 268.21
              vLLM              23.56                  945.40             21.59                 871.30
```

**Анализ результатов**

В статье описываем ключевые метрики

- **Transformers-Batch** показал наивысшую производительность (~7056 tokens/sec), опередив vLLM в **7.5 раз**
- **Наивный Transformers** значительно отстает (121 tokens/sec) - в **58 раз** медленнее лучшего результата
- **vLLM** демонстрирует высокую дисперсию между запусками (std = 21.59), что требует дополнительного исследования

**Скорость обработки (tokens/sec):**
```
1. Transformers-Batch: 7056.35 ± 268.21 tokens/sec
2. vLLM:              945.40 ± 871.30 tokens/sec  
3. Transformers:       120.95 ± 3.18 tokens/sec
```

**Время выполнения (time_seconds):**
```
1. Transformers-Batch: 2.26 ± 0.09 sec
2. vLLM:              23.56 ± 21.59 sec
3. Transformers:       124.02 ± 3.26 sec
```

**Статистическая значимость**

- **Transformers-Batch** показывает стабильные результаты (низкое std = 0.09)
- **vLLM** имеет аномально высокую вариативность - разница между запусками 38.82s vs 8.29s
- **Transformers** стабилен, но неэффективен

**Анализ аномалий vLLM**

Первые два запуска vLLM:
- Run 1: 38.82s (329 tokens/sec) - "холодный старт"?
- Run 2: 8.29s (1561 tokens/sec) - "прогретая" система

Гипотезы:
- Кеширование ключей/значений при повторных запусках
- Оптимизации memory allocation при последующих запросах
- JIT-компиляция в процессе работы

**Рекомендации** (идут в Discussion/Conclusion)

Для production:
```python
# Рекомендуемая конфигурация
ENGINE = "Transformers-Batch"
BATCHING_TYPE = ...
BASE_LLM = ...
GPU_USAGE = ...
```

**Для дальнейшего исследования:** (то, что описывается в Perspectives/Limitation)
1. Провести больше запусков vLLM для анализа стабильности
2. Исследовать причины "прогрева" vLLM
3. Протестировать на различных размерах батча
4. Проверить memory usage для каждого подхода

**Выводы** (Conclusion)

Здесь уместно привести рассуждения о причинах результатов

vLLM подходит для многопользовательских сценариев, потому что есть; требует дополнительной настройки, если мы хотим использовать его в качетсве инструмента оптимизации для создания прототипов

Наивный Transformers не подходит для production с точки зрения производительности